# 01 - Qubit Basics with Qiskit

This notebook introduces the core ideas from your first exercise:
- single-qubit circuit
- H (Hadamard) gate
- measurement
- Bell state circuit
- measurement distribution and shot count

## Concept: What Is a Single-Qubit Circuit?

A single qubit starts in $|0\rangle$. A circuit applies gates (operations) to transform that state, and then measurement maps the quantum state to a classical bit (`0` or `1`).

Without any gate, measuring a fresh qubit gives `0` with probability 1.

In [5]:
from qiskit import QuantumCircuit, transpile
from qiskit.providers.basic_provider import BasicSimulator

In [6]:
def run_counts(circuit: QuantumCircuit, shots: int = 1024) -> dict[str, int]:
    backend = BasicSimulator()
    compiled = transpile(circuit, backend)
    job = backend.run(compiled, shots=shots)
    return job.result().get_counts(compiled)

def to_probabilities(counts: dict[str, int]) -> dict[str, float]:
    total = sum(counts.values())
    return {k: v / total for k, v in sorted(counts.items())}

## Concept: H Gate (Hadamard)

The H gate puts a qubit into an equal superposition:
$$H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$$

After measurement, this gives approximately 50% `0` and 50% `1` over many shots.

In [ ]:
# QuantumCircuit(num_qubits, num_classical_bits)
single = QuantumCircuit(1, 1)

# Apply H to qubit index 0 (the only qubit in this circuit).
single.h(0)

# Measure qubit 0 and store the result in classical bit 0.
# Indices must be in range or Qiskit raises an index error.
single.measure(0, 0)

single.draw(output='text')

┌───┐┌─┐
  q: ┤ H ├┤M├
     └───┘└╥┘
c: 1/══════╩═
           0

In [8]:
single_counts = run_counts(single, shots=1024)
single_counts, to_probabilities(single_counts)

({'0': 502, '1': 522}, {'0': 0.490234375, '1': 0.509765625})

## Concept: Measurement and Distribution

A **shot** is one run of the circuit. Repeating a circuit many times builds a distribution of outcomes.

More shots reduce random fluctuation, so observed probabilities move closer to expected values.

In [ ]:
counts_100 = run_counts(single, shots=100)
counts_10000 = run_counts(single, shots=10000)

print('100 shots:', counts_100, to_probabilities(counts_100))
print('10000 shots:', counts_10000, to_probabilities(counts_10000))

## Concept: Bell State Circuit (Entanglement)

A Bell pair is created with H on qubit 0 followed by CX(0,1).

The resulting state is:
$$\frac{|00\rangle + |11\rangle}{\sqrt{2}}$$

So we expect mostly `00` and `11`, with `01` and `10` near zero on an ideal simulator.

In [9]:
# This circuit has 2 qubits and 2 classical bits.
bell = QuantumCircuit(2, 2)

# Put qubit 0 into superposition.
bell.h(0)

# Controlled-X: qubit 0 is control and qubit 1 is target.
bell.cx(0, 1)

# Measure qubits [0, 1] into classical bits [0, 1] one-to-one.
# All listed indices must exist in their respective registers.
bell.measure([0, 1], [0, 1])

bell.draw(output='text')

┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1

In [ ]:
bell_counts = run_counts(bell, shots=1024)
bell_counts, to_probabilities(bell_counts)

## Reflection

1. Why is the H-gate result close to 50/50 but not exactly 50/50?

Because we estimate probabilities from a finite number of shots. The true probabilities after applying $H$ to $|0\rangle$ are exactly $P(0)=P(1)=0.5$, but sampling noise causes small deviations in any finite run.

2. Why are `01` and `10` suppressed in the Bell-state experiment?

The Bell state created here is
$$
\frac{|00\rangle + |11\rangle}{\sqrt{2}}.
$$
This state has zero amplitude on $|01\rangle$ and $|10\rangle$, so on an ideal simulator those outcomes are absent (or near zero with noise/non-ideal effects).

3. What changed when moving from 100 to 10,000 shots?

The measured frequencies became more stable and closer to theoretical values. Increasing shots reduces statistical fluctuation, with typical error shrinking roughly like $1/\sqrt{N}$.

## Bonus: Is This Algebra? How Do We Calculate It?

Yes. Quantum states use **linear algebra**.

- A ket like $|0\rangle$ is a vector.
- Gates like $H$ are matrices.
- Applying a gate means matrix-vector multiplication.

For one qubit, the computational-basis vectors are:

$$
|0\rangle = \begin{bmatrix}1 \\ 0\end{bmatrix},\quad
|1\rangle = \begin{bmatrix}0 \\ 1\end{bmatrix}
$$

A superposition is a weighted sum, for example:

$$
|\psi\rangle = \alpha|0\rangle + \beta|1\rangle
$$

where $|\alpha|^2 + |\beta|^2 = 1$.

In [3]:
import numpy as np

# Basis kets as vectors
ket0 = np.array([[1.0], [0.0]])
ket1 = np.array([[0.0], [1.0]])

print("|0> =")
print(ket0)
print("\n|1> =")
print(ket1)

# A valid superposition: (|0> + |1>) / sqrt(2)
psi = (ket0 + ket1) / np.sqrt(2)
print("\n|psi> = (|0> + |1>) / sqrt(2) =")
print(psi)

# Probabilities are squared magnitudes of amplitudes
prob_0 = abs(psi[0, 0]) ** 2
prob_1 = abs(psi[1, 0]) ** 2
print(f"\nP(0)={prob_0:.3f}, P(1)={prob_1:.3f}, sum={prob_0 + prob_1:.3f}")

|0> =
[[1.]
 [0.]]

|1> =
[[0.]
 [1.]]

|psi> = (|0> + |1>) / sqrt(2) =
[[0.70710678]
 [0.70710678]]

P(0)=0.500, P(1)=0.500, sum=1.000


In [4]:
# Hadamard matrix
H = (1 / np.sqrt(2)) * np.array([[1.0, 1.0], [1.0, -1.0]])

# Apply H to |0>
h_on_0 = H @ ket0
print("H =")
print(H)
print("\nH|0> =")
print(h_on_0)

# This should match (|0> + |1>) / sqrt(2)
print("\nMatches (|0> + |1>) / sqrt(2)?", np.allclose(h_on_0, psi))

H =
[[ 0.70710678  0.70710678]
 [ 0.70710678 -0.70710678]]

H|0> =
[[0.70710678]
 [0.70710678]]

Matches (|0> + |1>) / sqrt(2)? True


## Bonus: How To Read the Text Circuit Diagram

When you run `circuit.draw(output='text')`, Qiskit prints an ASCII circuit map.

Example shape:

```text
     ┌───┐┌─┐
  q: ┤ X ├┤M├
     └───┘└╥┘
c: 1/══════╩═
           0
```

### Legend

- `q:`: Quantum wire/register label.
- `X`: Pauli-X gate (bit flip).
- `M`: Measurement operation.
- `c: 1/`: Classical register `c` with 1 classical bit.
- `0`: Classical bit index where the measurement is stored.
- Single horizontal lines (around `q`): quantum wires.
- Double horizontal lines (around `c`): classical wires.
- Vertical connector from `M` down to `c`: measurement result is sent from qubit wire to classical bit.

### Reading Direction

Read left to right. In this example:
1. Start on qubit `q`.
2. Apply `X`.
3. Measure with `M`.
4. Store the result in classical bit `c[0]`.